# 決算反応モデル — EDA

**実行環境**: Google Colab / Windows ローカル 自動判定
**BQアクセス**: 初回1回のみ → ローカルキャッシュ。2回目以降はBQアクセスゼロ。

**データソース**:
- BQ `STOCK.fin_summary` / J-Quants API — 決算実績・会社予想・配当・株式数
- BQ `STOCK.CONSENSUS` — コンセンサス（経常利益）
- BQ `STOCK.STOCK_PRICE_JQUANTS` — 株価（ラベル生成・時価総額計算）
- BQ `STOCK.STOCK_CODE_LIST` — セクター・市場区分
- BQ `STOCK.v_fin_summary_actual_for_q_on_q` — 前年同期実績（単独四半期値）

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
try:
    import japanize_matplotlib
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'], capture_output=True)
    import japanize_matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import requests
import warnings
warnings.filterwarnings('ignore')

# ── 実行環境判定 ──
try:
    from google.colab import auth, userdata
    RUNTIME = 'colab'
except ImportError:
    RUNTIME = 'local'

if RUNTIME == 'colab':
    # Colab: OAuth 認証
    auth.authenticate_user()
    from google.cloud import bigquery
    bq = bigquery.Client(project='gmailpj-357912')
    JQUANTS_API_KEY = userdata.get('JQUANTS_API_KEY')
else:
    # ローカル (Windows): SSL workaround + サービスアカウント
    import urllib3, requests as _req
    from requests.adapters import HTTPAdapter as _HA
    urllib3.disable_warnings()
    class _NoVerify(_HA):
        def send(self, req, **kw): kw['verify'] = False; return super().send(req, **kw)
    _orig = _req.Session.__init__
    def _p(self, *a, **kw): _orig(self, *a, **kw); self.mount('https://', _NoVerify()); self.verify = False
    _req.Session.__init__ = _p

    PROJECT_ROOT = Path(r'C:\gdrive\claude\investment-agent')
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')

    from google.cloud import bigquery
    from google.oauth2 import service_account
    KEY_FILE = str(PROJECT_ROOT / 'keys' / 'gcp-service-account.json')
    creds = service_account.Credentials.from_service_account_file(KEY_FILE)
    bq = bigquery.Client(credentials=creds, project='gmailpj-357912')
    JQUANTS_API_KEY = os.environ['JQUANTS_API_KEY']

JQUANTS_HEADERS = {'x-api-key': JQUANTS_API_KEY}
JQUANTS_BASE_URL = 'https://api.jquants.com/v2'


def jquants_get(endpoint: str, params: dict) -> list[dict]:
    """J-Quants API V2 GET（ページネーション・レートリミット対応）."""
    all_records, params = [], dict(params)
    while True:
        time.sleep(0.5)
        resp = requests.get(f'{JQUANTS_BASE_URL}{endpoint}',
                            headers=JQUANTS_HEADERS, params=params)
        if resp.status_code == 429:
            print('  Rate limited, waiting 30s...')
            time.sleep(30)
            continue
        if resp.status_code != 200:
            print(f'  Error {resp.status_code}: {resp.text[:200]}')
            break
        body = resp.json()
        all_records.extend(body.get('data', []))
        pk = body.get('pagination_key')
        if not pk:
            break
        params['pagination_key'] = pk
    return all_records

# Gemini Flash（優待の拡充/改悪判定用）
from google import genai
if RUNTIME == 'colab':
    gemini_client = genai.Client(vertexai=True, project='gmailpj-357912', location='us-central1')
else:
    gemini_client = genai.Client(
        vertexai=True, project='gmailpj-357912', location='us-central1',
        credentials=creds)
GEMINI_MODEL = 'gemini-2.5-flash'

print(f'Setup OK (runtime={RUNTIME})')

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────
# ▼▼▼ ここを変更して使う ▼▼▼
DATE_FROM = '20240101'   # 決算発表日 開始 (YYYYMMDD)
DATE_TO   = '20260403'   # 決算発表日 終了 (YYYYMMDD)
USE_BQ    = True         # True=BQ（過去データ）/ False=J-Quants API（BQ未収録の最新データ）
FORCE_RELOAD = True     # True=BQから強制再ダウンロード / False=ローカルキャッシュ優先
# ▲▲▲ ここを変更して使う ▲▲▲

# キャッシュディレクトリ（環境自動判定）
if RUNTIME == 'colab':
    CACHE_DIR = Path('/content/earnings_model_cache')
else:
    CACHE_DIR = Path(r'C:\tmp\earnings_model_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# pandas 表示設定
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. データ取得（BQ → ローカルキャッシュ）

- **初回**: BQ から全5テーブルを1回でダウンロード → キャッシュディレクトリに CSV 保存
  - Colab: `/content/earnings_model_cache/`
  - ローカル: `C:	mp\earnings_model_cache\`
- **2回目以降**: ローカル CSV から読み込み（**BQ アクセスゼロ**）
- `FORCE_RELOAD=True` で強制再ダウンロード

In [ ]:
# ── データ取得 + キャッシュ ──────────────────────────────────────
_tag = f'{DATE_FROM}_{DATE_TO}'
_CACHE = {
    'fin_summary': CACHE_DIR / f'{_tag}_fin_summary.csv',
    'sector':      CACHE_DIR / 'sector_master.csv',
    'consensus':   CACHE_DIR / f'{_tag}_consensus.csv',
    'price':       CACHE_DIR / f'{_tag}_price.csv',
    'yoy':          CACHE_DIR / f'{_tag}_yoy.csv',
    'prev_forecast': CACHE_DIR / f'{_tag}_prev_forecast.csv',
    'prev_year_q':  CACHE_DIR / f'{_tag}_prev_year_q.csv',
    'tdnet_events': CACHE_DIR / f'{_tag}_tdnet_events.csv',
}


def _all_cached() -> bool:
    return all(p.exists() for p in _CACHE.values())


def _download_fin_summary_bq(d_from: str, d_to: str) -> pd.DataFrame:
    """BQ fin_summary から取得（修正決算は最新を優先）."""
    return bq.query(f"""
        WITH ranked AS (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY LOCAL_CODE, TYPE_OF_CURRENT_PERIOD, CURRENT_FISCAL_YEAR_END_DATE
                ORDER BY DISCLOSED_DATE DESC
            ) AS rn
            FROM `gmailpj-357912.STOCK.fin_summary`
            WHERE DISCLOSED_DATE BETWEEN '{d_from}' AND '{d_to}'
        )
        SELECT
            DISCLOSED_DATE AS DiscDate, DISCLOSED_TIME AS DiscTime,
            LOCAL_CODE AS Code, DISCLOSURE_NUMBER AS DiscNo,
            TYPE_OF_DOCUMENT AS DocType, TYPE_OF_CURRENT_PERIOD AS CurPerType,
            CURRENT_FISCAL_YEAR_END_DATE AS CurFYEn,
            NET_SALES AS Sales, OPERATING_PROFIT AS OP, ORDINARY_PROFIT AS OdP, PROFIT AS NP,
            EARNINGS_PER_SHARE AS EPS, TOTAL_ASSETS AS TA, EQUITY AS Eq,
            CASH_FLOWS_FROM_OPERATING_ACTIVITIES AS CFO,
            RESULT_DIVIDEND_PER_SHARE_ANNUAL AS DivAnn,
            FORECAST_DIVIDEND_PER_SHARE_ANNUAL AS FDivAnn,
            FORECAST_PAYOUT_RATIO_ANNUAL AS FPayoutRatioAnn,
            FORECAST_NET_SALES AS FSales, FORECAST_OPERATING_PROFIT AS FOP,
            FORECAST_ORDINARY_PROFIT AS FOdP, FORECAST_PROFIT AS FNP,
            NEXT_YEAR_FORECAST_NET_SALES AS NxFSales,
            NEXT_YEAR_FORECAST_OPERATING_PROFIT AS NxFOP,
            NEXT_YEAR_FORECAST_ORDINARY_PROFIT AS NxFOdP,
            NEXT_YEAR_FORECAST_PROFIT AS NxFNp,
            NUMBER_OF_ISSUED_AND_OUTSTANDING_SHARES_AT_THE_END_OF_FISCAL_YEAR_INCLUDING_TREASURY_STOCK AS ShOutFY,
            NUMBER_OF_TREASURY_STOCK_AT_THE_END_OF_FISCAL_YEAR AS TrShFY,
            AVERAGE_NUMBER_OF_SHARES AS AvgSh
        FROM ranked WHERE rn = 1
        ORDER BY DiscDate, Code
    """).to_dataframe()


def _download_fin_summary_api(date_from: str, date_to: str) -> pd.DataFrame:
    """J-Quants API から fin_summary を取得（BQ未収録の最新データ用）."""
    d_from = datetime.strptime(date_from, '%Y%m%d')
    d_to   = datetime.strptime(date_to,   '%Y%m%d')
    all_records, d = [], d_from
    while d <= d_to:
        recs = jquants_get('/fins/summary', {'date': d.strftime('%Y%m%d')})
        if recs:
            print(f'  {d:%Y-%m-%d}: {len(recs)} 件')
            all_records.extend(recs)
        d += timedelta(days=1)
    if not all_records:
        return pd.DataFrame()
    df = pd.DataFrame(all_records).replace('', np.nan)
    df['Code'] = df['Code'].str[:4]
    str_cols = {'Code','DiscNo','DocType','CurPerType','DiscTime',
                'CurPerSt','CurPerEn','CurFYSt','CurFYEn','NxtFYSt','NxtFYEn',
                'MatChgSub','SigChgInC','ChgByASRev','ChgNoASRev','ChgAcEst','RetroRst'}
    for col in df.columns:
        if col not in str_cols and col != 'DiscDate':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


def _download_all():
    """BQ/API から全データをダウンロード → CSV キャッシュ保存."""
    d_from = f'{DATE_FROM[:4]}-{DATE_FROM[4:6]}-{DATE_FROM[6:]}'
    d_to   = f'{DATE_TO[:4]}-{DATE_TO[4:6]}-{DATE_TO[6:]}'
    d_from_ext = (datetime.strptime(DATE_FROM, '%Y%m%d') - timedelta(days=5)).strftime('%Y-%m-%d')
    d_to_ext   = (datetime.strptime(DATE_TO, '%Y%m%d') + timedelta(days=5)).strftime('%Y-%m-%d')

    # 1. fin_summary
    if USE_BQ:
        print('[1/5] fin_summary (BQ) ...', end=' ', flush=True)
        df = _download_fin_summary_bq(d_from, d_to)
    else:
        print('[1/5] fin_summary (API) ...')
        df = _download_fin_summary_api(DATE_FROM, DATE_TO)
    df.to_csv(_CACHE['fin_summary'], index=False)
    print(f'{len(df)} 件')

    # 2. セクターマスタ
    print('[2/5] sector_master (BQ) ...', end=' ', flush=True)
    df = bq.query("""
        SELECT TICKER, STOCK_NAME, MARKET_CATEGORY,
               INDUSTRY_33_CODE, INDUSTRY_33_CATEGORY, SIZE_CODE, SIZE_CATEGORY
        FROM `gmailpj-357912.STOCK.STOCK_CODE_LIST`
    """).to_dataframe()
    df.to_csv(_CACHE['sector'], index=False)
    print(f'{len(df)} 件')

    # 3. コンセンサス（発表日以前の最新1件/銘柄別）
    print('[3/5] consensus (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        WITH ranked AS (
            SELECT TICKER, DATAAT, FY, QUARTER, PROFIT,
                   ROW_NUMBER() OVER (PARTITION BY TICKER ORDER BY DATAAT DESC) AS rn
            FROM `gmailpj-357912.STOCK.CONSENSUS`
            WHERE DATAAT <= '{d_to}'
        )
        SELECT TICKER, DATAAT, FY, QUARTER, PROFIT AS CONSENSUS_PROFIT
        FROM ranked WHERE rn = 1
    """).to_dataframe()
    df.to_csv(_CACHE['consensus'], index=False)
    print(f'{len(df)} 件')

    # 4. 株価（発表日前後5日分）
    print('[4/5] stock_price (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        SELECT DATE, TICKER, ADJ_OPEN, ADJ_CLOSE, VOLUME
        FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS`
        WHERE DATE BETWEEN '{d_from_ext}' AND '{d_to_ext}'
          AND IS_PREFERRED = FALSE
        ORDER BY TICKER, DATE
    """).to_dataframe()
    df.to_csv(_CACHE['price'], index=False)
    print(f'{len(df)} 件')

    # 5. Q単独実績（前年同期比用）
    print('[5/5] yoy (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        SELECT LOCAL_CODE AS Code, DISCLOSED_DATE AS DiscDate, QUARTER,
               NET_SALES AS Q_Sales, OPERATING_PROFIT AS Q_OP,
               ORDINARY_PROFIT AS Q_OdP, PROFIT AS Q_NP
        FROM `gmailpj-357912.STOCK.v_fin_summary_actual_for_q_on_q`
        WHERE DISCLOSED_DATE BETWEEN '{d_from}' AND '{d_to}'
          AND TYPE_OF_DOCUMENT LIKE '%Consolidated%'
    """).to_dataframe()
    df.to_csv(_CACHE['yoy'], index=False)
    print(f'{len(df)} 件')
    # 6. 前回発表時の会社予想（同一会計年度の前四半期レコード）
    print('[6/8] prev_forecast (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        WITH target AS (
            SELECT DISTINCT LOCAL_CODE, CURRENT_FISCAL_YEAR_END_DATE, TYPE_OF_CURRENT_PERIOD
            FROM `gmailpj-357912.STOCK.fin_summary`
            WHERE DISCLOSED_DATE BETWEEN '{d_from}' AND '{d_to}'
              AND TYPE_OF_DOCUMENT LIKE '%FinancialStatements%'
        ),
        prev_records AS (
            SELECT
                t.LOCAL_CODE AS Code,
                t.TYPE_OF_CURRENT_PERIOD AS target_period,
                t.CURRENT_FISCAL_YEAR_END_DATE AS CurFYEn,
                f.NET_SALES AS prev_Sales,
                f.OPERATING_PROFIT AS prev_OP,
                f.PROFIT AS prev_NP,
                f.FORECAST_NET_SALES AS prev_FSales,
                f.FORECAST_OPERATING_PROFIT AS prev_FOP,
                f.FORECAST_PROFIT AS prev_FNP,
                ROW_NUMBER() OVER (
                    PARTITION BY t.LOCAL_CODE, t.CURRENT_FISCAL_YEAR_END_DATE, t.TYPE_OF_CURRENT_PERIOD
                    ORDER BY f.DISCLOSED_DATE DESC
                ) AS rn
            FROM target t
            INNER JOIN `gmailpj-357912.STOCK.fin_summary` f
                ON f.LOCAL_CODE = t.LOCAL_CODE
                AND f.CURRENT_FISCAL_YEAR_END_DATE = t.CURRENT_FISCAL_YEAR_END_DATE
                AND f.TYPE_OF_CURRENT_PERIOD = CASE t.TYPE_OF_CURRENT_PERIOD
                    WHEN '2Q' THEN '1Q'
                    WHEN '3Q' THEN '2Q'
                    WHEN 'FY' THEN '3Q'
                END
        )
        SELECT Code, target_period, CurFYEn,
               prev_Sales, prev_OP, prev_NP,
               prev_FSales, prev_FOP, prev_FNP
        FROM prev_records WHERE rn = 1
    """).to_dataframe()
    df.to_csv(_CACHE['prev_forecast'], index=False)
    print(f'{len(df)} 件')

    # 7. 前年同期Q単独実績（YoY計算用）
    print('[7/8] prev_year_q (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        WITH cur AS (
            SELECT LOCAL_CODE, QUARTER, CURRENT_FISCAL_YEAR_START_DATE, DISCLOSED_DATE
            FROM `gmailpj-357912.STOCK.v_fin_summary_actual_for_q_on_q`
            WHERE DISCLOSED_DATE BETWEEN '{d_from}' AND '{d_to}'
              AND TYPE_OF_DOCUMENT LIKE '%Consolidated%'
        )
        SELECT
            cur.LOCAL_CODE AS Code,
            cur.DISCLOSED_DATE AS DiscDate,
            cur.QUARTER,
            p.NET_SALES AS PrevY_Q_Sales,
            p.OPERATING_PROFIT AS PrevY_Q_OP,
            p.ORDINARY_PROFIT AS PrevY_Q_OdP,
            p.PROFIT AS PrevY_Q_NP
        FROM cur
        JOIN `gmailpj-357912.STOCK.v_fin_summary_actual_for_q_on_q` p
            ON cur.LOCAL_CODE = p.LOCAL_CODE
            AND cur.QUARTER = p.QUARTER
            AND DATE_ADD(p.CURRENT_FISCAL_YEAR_START_DATE, INTERVAL 1 YEAR)
                = cur.CURRENT_FISCAL_YEAR_START_DATE
        WHERE p.TYPE_OF_DOCUMENT LIKE '%Consolidated%'
    """).to_dataframe()
    df.to_csv(_CACHE['prev_year_q'], index=False)
    print(f'{len(df)} 件')

    # 8. TDnetイベント（自社株買い・株式分割・優待、決算同時アナウンス）
    print('[8/8] tdnet_events (BQ) ...', end=' ', flush=True)
    df = bq.query(f"""
        SELECT
            TICKER AS Code,
            SUBMISSION_DATE,
            MAIN_CATEGORY,
            DOC_TITLE,
            CHUNK_TEXT
        FROM `gmailpj-357912.STOCK.TDNET_DOCUMENTS_ENHANCED`
        WHERE (MAIN_CATEGORY IN ('自己株式取得', '株式分割・併合')
               OR DOC_TITLE LIKE '%株主優待%')
          AND SUBMISSION_DATE BETWEEN '{d_from}' AND '{d_to}'
    """).to_dataframe()
    df.to_csv(_CACHE['tdnet_events'], index=False)
    print(f'{len(df)} 件')

    print(f'\nキャッシュ保存完了: {CACHE_DIR}')


def _load_all():
    """ローカル CSV から全データを読み込み."""
    df_raw = pd.read_csv(_CACHE['fin_summary'])
    df_raw['DiscDate'] = pd.to_datetime(df_raw['DiscDate']).dt.date
    df_raw['Code'] = df_raw['Code'].astype(str)

    df_sector = pd.read_csv(_CACHE['sector'])
    df_sector['TICKER'] = df_sector['TICKER'].astype(str)

    df_consensus = pd.read_csv(_CACHE['consensus'])
    df_consensus['TICKER'] = df_consensus['TICKER'].astype(str)
    if 'DATAAT' in df_consensus.columns:
        df_consensus['DATAAT'] = pd.to_datetime(df_consensus['DATAAT']).dt.date

    df_price = pd.read_csv(_CACHE['price'])
    df_price['TICKER'] = df_price['TICKER'].astype(str)
    df_price['DATE'] = pd.to_datetime(df_price['DATE']).dt.date

    df_yoy = pd.read_csv(_CACHE['yoy'])
    df_yoy['Code'] = df_yoy['Code'].astype(str)
    df_yoy['DiscDate'] = pd.to_datetime(df_yoy['DiscDate']).dt.date

    df_prev_fc = pd.read_csv(_CACHE['prev_forecast'])
    df_prev_fc['Code'] = df_prev_fc['Code'].astype(str)

    df_prev_year_q = pd.read_csv(_CACHE['prev_year_q'])
    df_prev_year_q['Code'] = df_prev_year_q['Code'].astype(str)
    if 'DiscDate' in df_prev_year_q.columns:
        df_prev_year_q['DiscDate'] = pd.to_datetime(df_prev_year_q['DiscDate']).dt.date

    df_tdnet_events = pd.read_csv(_CACHE['tdnet_events'])
    if len(df_tdnet_events) > 0:
        df_tdnet_events['Code'] = df_tdnet_events['Code'].astype(str)
        if 'SUBMISSION_DATE' in df_tdnet_events.columns:
            df_tdnet_events['SUBMISSION_DATE'] = pd.to_datetime(df_tdnet_events['SUBMISSION_DATE']).dt.date

    return df_raw, df_sector, df_consensus, df_price, df_yoy, df_prev_fc, df_prev_year_q, df_tdnet_events


# ── 実行 ──────────────────────────────────────────────────────
if FORCE_RELOAD or not _all_cached():
    _download_all()
else:
    print(f'キャッシュ使用（BQアクセスなし）: {CACHE_DIR}')

df_raw, df_sector, df_consensus, df_price, df_yoy, df_prev_fc, df_prev_year_q, df_tdnet_events = _load_all()
print(f'\nfin_summary: {len(df_raw)}, sector: {len(df_sector)}, '
      f'consensus: {len(df_consensus)}, price: {len(df_price)}, yoy: {len(df_yoy)}')
print(f'prev_forecast: {len(df_prev_fc)}, prev_year_q: {len(df_prev_year_q)}, tdnet_events: {len(df_tdnet_events)}')

In [ ]:
# DocType（開示書類種別）の分布を確認
print(df_raw['DocType'].value_counts())
print()
# CurPerType（四半期種別）の分布
print(df_raw['CurPerType'].value_counts())

In [ ]:
# ── フィルタリング ──────────────────────────────────────────────
# 決算実績発表のみを分析対象にする（業績修正・配当修正は除外）
# 業績修正/配当修正は「事前に修正があったか」の特徴量として使う

# 決算実績 = DocType に "FinancialStatements" を含むもの
is_financial_stmt = df_raw["DocType"].str.contains("FinancialStatements", na=False)

# 事前修正フラグ用: 業績修正・配当修正を別途保持
df_revisions = df_raw[~is_financial_stmt].copy()
print(f"除外（業績修正・配当修正等）: {len(df_revisions)} 件")
print(df_revisions["DocType"].value_counts().to_string())

# 分析対象 = 決算実績のみ
df_earn = df_raw[is_financial_stmt].copy().reset_index(drop=True)
print(f"\n分析対象（決算実績）: {len(df_earn)} 件")
print(df_earn["DocType"].value_counts().to_string())

# ── 事前修正フラグ生成 ──
# 同一 Code で DiscDate 以前に業績修正 or 配当修正が存在するか
# （同日の修正も「事前」に含める＝決算と同時に修正を出すケースがあるため）
def _build_prior_flag(df_target, df_rev, doc_pattern, col_name):
    """df_target の各行に対し、同Code で DiscDate 以前に doc_pattern を含む修正があるかフラグ付与."""
    rev_subset = df_rev[df_rev["DocType"].str.contains(doc_pattern, na=False)]
    if len(rev_subset) == 0:
        df_target[col_name] = False
        return
    # Code × DiscDate で最も早い修正日を取得
    rev_dates = rev_subset.groupby("Code")["DiscDate"].min().reset_index()
    rev_dates.columns = ["Code", "_rev_date"]
    merged = df_target[["Code", "DiscDate"]].merge(rev_dates, on="Code", how="left")
    df_target[col_name] = merged["_rev_date"].notna() & (merged["_rev_date"] <= merged["DiscDate"])

_build_prior_flag(df_earn, df_revisions, "EarnForecastRevision", "has_prior_revision")
_build_prior_flag(df_earn, df_revisions, "DividendForecastRevision", "has_prior_div_revision")

print(f"\n事前業績修正あり: {df_earn['has_prior_revision'].sum()} 件")
print(f"事前配当修正あり: {df_earn['has_prior_div_revision'].sum()} 件")
df_earn[["Code", "DiscDate", "DocType", "CurPerType", "has_prior_revision", "has_prior_div_revision"]].head(10)

## 2. データ結合

In [ ]:
# ── 発表タイミング判定 + 株価反応ラベル生成 ─────────────────────────────────
# 2024-11-05 から東証取引終了が 15:00→15:30 に変更
# DiscTime < 取引終了時刻 → ザラ場発表 → 発表当日の株価変動を観測
# DiscTime >= 取引終了時刻 or NULL → 引け後発表 → 翌営業日の株価変動を観測

df_price_sorted = df_price.sort_values(["TICKER", "DATE"])
df_price_ext = df_price_sorted.copy()
df_price_ext["PREV_ADJ_CLOSE"] = df_price_ext.groupby("TICKER")["ADJ_CLOSE"].shift(1)
df_price_ext["NEXT_DATE"]      = df_price_ext.groupby("TICKER")["DATE"].shift(-1)
df_price_ext["NEXT_ADJ_CLOSE"] = df_price_ext.groupby("TICKER")["ADJ_CLOSE"].shift(-1)
df_price_ext["NEXT_ADJ_OPEN"]  = df_price_ext.groupby("TICKER")["ADJ_OPEN"].shift(-1)

_CLOSE_TIME_CHANGE_DATE = "2024-11-05"
def _is_intraday(disc_date, disc_time) -> bool:
    if pd.isna(t) or str(t).strip() in ("", "nan", "None"):
        return False
    close = "15:30" if str(disc_date) >= _CLOSE_TIME_CHANGE_DATE else "15:00"
    return str(disc_time)[:5] < close

df_earn["is_intraday"] = df_earn.apply(lambda r: _is_intraday(r["DiscDate"], r["DiscTime"]), axis=1)

price_on_disc = df_price_ext[["TICKER", "DATE", "ADJ_CLOSE", "ADJ_OPEN",
                               "PREV_ADJ_CLOSE", "NEXT_DATE", "NEXT_ADJ_CLOSE", "NEXT_ADJ_OPEN"]]

df_earn_price = df_earn[["Code", "DiscDate", "is_intraday"]].merge(
    price_on_disc.rename(columns={"TICKER": "Code", "DATE": "_disc_date_key"}),
    left_on=["Code", "DiscDate"],
    right_on=["Code", "_disc_date_key"],
    how="left"
).drop(columns=["_disc_date_key"])

df_earn_price["price_date"] = df_earn_price["NEXT_DATE"].where(
    ~df_earn_price["is_intraday"], df_earn_price["DiscDate"])
df_earn_price["base_close"] = df_earn_price["ADJ_CLOSE"].where(
    ~df_earn_price["is_intraday"], df_earn_price["PREV_ADJ_CLOSE"])
df_earn_price["react_close"] = df_earn_price["NEXT_ADJ_CLOSE"].where(
    ~df_earn_price["is_intraday"], df_earn_price["ADJ_CLOSE"])
df_earn_price["react_open"] = df_earn_price["NEXT_ADJ_OPEN"].where(
    ~df_earn_price["is_intraday"], df_earn_price["ADJ_OPEN"])

df_earn_price["LABEL_CLOSE_RETURN"] = df_earn_price["react_close"] / df_earn_price["base_close"] - 1
df_earn_price["LABEL_OPEN_RETURN"]  = df_earn_price["react_open"]  / df_earn_price["base_close"] - 1

df_price_label = df_earn_price[[
    "Code", "DiscDate", "price_date",
    "ADJ_CLOSE", "LABEL_CLOSE_RETURN", "LABEL_OPEN_RETURN"
]].copy()

# 同一 Code+DiscDate の重複除去（株価は四半期によらず同一なので先頭を採用）
df_price_label = df_price_label.drop_duplicates(subset=["Code", "DiscDate"], keep="first")

print(f"ザラ場発表: {df_earn['is_intraday'].sum()} 件 / 引け後発表: {(~df_earn['is_intraday']).sum()} 件")
print(f"ラベル欠損率: close={df_price_label['LABEL_CLOSE_RETURN'].isnull().mean():.1%}, open={df_price_label['LABEL_OPEN_RETURN'].isnull().mean():.1%}")
df_price_label[["Code", "DiscDate", "price_date", "LABEL_CLOSE_RETURN", "LABEL_OPEN_RETURN"]].head(10)

In [ ]:
# ── 全テーブル結合
df = df_earn.copy()
df = df.merge(df_sector, left_on="Code", right_on="TICKER", how="left")

# コンセンサス: PROFIT は百万円単位 → 円に変換して結合
df_cons_yen = df_consensus[["TICKER", "CONSENSUS_PROFIT", "DATAAT"]].copy()
df_cons_yen["CONSENSUS_PROFIT"] = df_cons_yen["CONSENSUS_PROFIT"] * 1_000_000  # 百万円→円
df = df.merge(df_cons_yen, left_on="Code", right_on="TICKER", how="left")

df = df.merge(
    df_price_label[["Code", "DiscDate", "price_date",
                    "ADJ_CLOSE", "LABEL_CLOSE_RETURN", "LABEL_OPEN_RETURN"]],
    on=["Code", "DiscDate"], how="left"
)
# yoy の QUARTER → CurPerType マッピング (4Q→FY)
df_yoy_m = df_yoy.copy()
df_yoy_m["CurPerType"] = df_yoy_m["QUARTER"].replace({"4Q": "FY"})
df = df.merge(
    df_yoy_m[["Code", "DiscDate", "CurPerType", "Q_Sales", "Q_OP", "Q_OdP", "Q_NP"]],
    on=["Code", "DiscDate", "CurPerType"], how="left"
)

print(f"結合後: {len(df)} 件, カラム数: {len(df.columns)}")
print("ザラ場/引け後:"); print(df["is_intraday"].value_counts())
print("ラベル欠損率:"); print(df[["LABEL_CLOSE_RETURN", "LABEL_OPEN_RETURN"]].isnull().mean())
print("Q単独実績欠損率:"); print(df[["Q_Sales", "Q_OP", "Q_OdP", "Q_NP"]].isnull().mean())
print("コンセンサス一致率:"); print(f"{df['CONSENSUS_PROFIT'].notna().mean():.1%}")

# 前回発表時の会社予想（同一会計年度の前四半期レコード）
df = df.merge(
    df_prev_fc.rename(columns={"target_period": "CurPerType"}),
    on=["Code", "CurPerType", "CurFYEn"], how="left"
)

# 前年同期Q単独実績（YoY計算用）
# prev_year_q の QUARTER → CurPerType マッピング (4Q→FY)
df_prev_year_q_m = df_prev_year_q.copy()
df_prev_year_q_m["CurPerType"] = df_prev_year_q_m["QUARTER"].replace({"4Q": "FY"})
df = df.merge(
    df_prev_year_q_m[["Code", "DiscDate", "CurPerType", "PrevY_Q_Sales", "PrevY_Q_OP", "PrevY_Q_OdP", "PrevY_Q_NP"]],
    on=["Code", "DiscDate", "CurPerType"], how="left"
)

# ── TDnetイベント（決算発表と同時アナウンス）────────────────────
import re

# (a) 自社株買い
def _extract_buyback(df_ev, code, disc_date):
    """決算同日の自社株買いアナウンスから割合(%)を抽出."""
    if len(df_ev) == 0: return False, np.nan
    sub = df_ev[(df_ev["Code"] == code) & (df_ev["SUBMISSION_DATE"] == disc_date)
              & (df_ev["MAIN_CATEGORY"] == "自己株式取得")]
    if len(sub) == 0: return False, np.nan
    for txt in sub["CHUNK_TEXT"].dropna():
        m = re.search(r"割合[^\d]*([\d.]+)\s*[%％]", str(txt))
        if m: return True, float(m.group(1))
    return True, np.nan

# (b) 株式分割
def _extract_split(df_ev, code, disc_date):
    """決算同日の株式分割アナウンスから分割比率を抽出."""
    if len(df_ev) == 0: return False, np.nan
    sub = df_ev[(df_ev["Code"] == code) & (df_ev["SUBMISSION_DATE"] == disc_date)
              & (df_ev["MAIN_CATEGORY"] == "株式分割・併合")]
    if len(sub) == 0: return False, np.nan
    for txt in sub["CHUNK_TEXT"].dropna():
        # "1株につき2株" or "1:2" パターン
        m = re.search(r"1\s*株につき\s*([\d.]+)\s*株", str(txt))
        if m: return True, float(m.group(1))
        m = re.search(r"1\s*[:：]\s*([\d.]+)", str(txt))
        if m: return True, float(m.group(1))
    return True, np.nan

# (c) 株主優待（Gemini Flash で拡充/改悪を判定）
def _classify_yutai(df_ev, code, disc_date):
    """決算同日の優待アナウンスをGemini Flashで分類.
    Returns: (has_yutai, score)  score: +1=拡充/新設, -1=改悪/廃止, 0=変更なし/不明
    """
    if len(df_ev) == 0: return False, 0
    sub = df_ev[(df_ev["Code"] == code) & (df_ev["SUBMISSION_DATE"] == disc_date)
              & (df_ev["DOC_TITLE"].str.contains("株主優待", na=False))]
    if len(sub) == 0: return False, 0
    # 最初のドキュメントのタイトル+テキスト冒頭を送信
    row = sub.iloc[0]
    title = str(row.get("DOC_TITLE", ""))
    text = str(row.get("CHUNK_TEXT", ""))[:500]
    prompt = (
        "以下のTDnet適時開示は株主優待に関するものです。\n"
        "内容を読んで、以下の1語のみで回答してください:\n"
        "拡充 / 新設 / 改悪 / 廃止 / 不明\n\n"
        f"タイトル: {title}\nテキスト: {text}"
    )
    try:
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL, contents=prompt)
        answer = resp.text.strip()
        if "拡充" in answer or "新設" in answer: return True, 1
        if "改悪" in answer or "廃止" in answer: return True, -1
        return True, 0
    except Exception as e:
        print(f"  Gemini error ({code}): {e}")
        return True, 0

# 全行に適用
bb_results = df.apply(lambda r: _extract_buyback(df_tdnet_events, r["Code"], r["DiscDate"]), axis=1)
df["has_buyback"] = bb_results.apply(lambda x: x[0])
df["buyback_pct"] = bb_results.apply(lambda x: x[1])

sp_results = df.apply(lambda r: _extract_split(df_tdnet_events, r["Code"], r["DiscDate"]), axis=1)
df["has_stock_split"] = sp_results.apply(lambda x: x[0])
df["split_ratio"] = sp_results.apply(lambda x: x[1])

yu_results = df.apply(lambda r: _classify_yutai(df_tdnet_events, r["Code"], r["DiscDate"]), axis=1)
df["has_yutai"] = yu_results.apply(lambda x: x[0])
df["yutai_score"] = yu_results.apply(lambda x: x[1])  # +1=拡充, -1=改悪, 0=不明

print(f"前回予想一致率: {df['prev_FSales'].notna().mean():.1%}")
print(f"前年同期一致率: {df['PrevY_Q_Sales'].notna().mean():.1%}")
print(f"自社株買いあり: {df['has_buyback'].sum()} 件")
print(f"株式分割あり: {df['has_stock_split'].sum()} 件")
print(f"優待変更あり: {df['has_yutai'].sum()} 件 (拡充={( df['yutai_score']==1).sum()}, 改悪={(df['yutai_score']==-1).sum()})")

# 安全策: merge後の重複除去
_before = len(df)
df = df.drop_duplicates(subset=["Code", "DiscDate", "CurPerType"], keep="first").reset_index(drop=True)
if len(df) < _before:
    print(f"重複除去: {_before} -> {len(df)} ({_before - len(df)} 行削除)")


## 3. 特徴量エンジニアリング

In [ ]:
def safe_ratio(numerator, denominator, clip: float = 5.0) -> pd.Series:
    denom = denominator.abs().replace(0, np.nan)
    return ((numerator - denominator) / denom).clip(-clip, clip)


REMAINING_Q = {"1Q": 3, "2Q": 2, "3Q": 1, "FY": 0}
remaining_q = df["CurPerType"].map(REMAINING_Q)
is_fy = df["CurPerType"] == "FY"

feat = pd.DataFrame(index=df.index)
feat["Code"]       = df["Code"]
feat["DiscDate"]   = df["DiscDate"]
feat["CurPerType"] = df["CurPerType"]
feat["DocType"]    = df["DocType"]

# 3-1. Q単独実績 vs 会社予想（Q単独ペース換算）
implied_q_sales = (df["FSales"] - df["Sales"]) / remaining_q.replace(0, np.nan)
implied_q_op    = (df["FOP"]    - df["OP"])    / remaining_q.replace(0, np.nan)
implied_q_np    = (df["FNP"]    - df["NP"])    / remaining_q.replace(0, np.nan)

feat["surprise_sales_vs_forecast"] = np.where(
    is_fy, safe_ratio(df["Sales"], df["FSales"]),
    safe_ratio(df["Q_Sales"], implied_q_sales))
feat["surprise_op_vs_forecast"] = np.where(
    is_fy, safe_ratio(df["OP"], df["FOP"]),
    safe_ratio(df["Q_OP"], implied_q_op))
feat["surprise_np_vs_forecast"] = np.where(
    is_fy, safe_ratio(df["NP"], df["FNP"]),
    safe_ratio(df["Q_NP"], implied_q_np))

# 3-2. 実績（累積）vs コンセンサス（累積、円換算済み）
feat["surprise_odp_vs_consensus"] = safe_ratio(df["OdP"], df["CONSENSUS_PROFIT"])

# 3-3. 翌期予想 vs 今期実績比（FY のみ有意）
feat["guidance_fy_sales"] = safe_ratio(df["NxFSales"], df["Sales"])
feat["guidance_fy_op"]    = safe_ratio(df["NxFOP"],    df["OP"])
feat["guidance_fy_np"]    = safe_ratio(df["NxFNp"],    df["NP"])

# 3-3b. 残りQ期間の会社予想ペース（Q1-Q3で有効）
feat["guidance_remaining_q_op"] = implied_q_op

# 3-3c. 通期予想進捗率（累計実績 / 通期予想）— 高進捗+据え置き = 出尽くしリスク
feat["progress_sales"] = np.where(df["FSales"].abs() > 0, df["Sales"] / df["FSales"], np.nan)
feat["progress_op"]    = np.where(df["FOP"].abs() > 0, df["OP"] / df["FOP"], np.nan)
feat["progress_np"]    = np.where(df["FNP"].abs() > 0, df["NP"] / df["FNP"], np.nan)

# 3-3d. ガイダンス修正有無フラグ（今回発表で通期予想を修正したか）
# prev_FSales が存在し、かつ現在の FSales と異なれば修正あり
feat["has_guidance_revision"] = (
    df["prev_FSales"].notna() &
    (df["FSales"] != df["prev_FSales"])
).astype(int)

# 3-4. 配当変化
feat["dividend_surprise"] = safe_ratio(df["DivAnn"], df["FDivAnn"], clip=3.0)

# 3-5. 時価総額（発表日終値 × 浮動株数）
float_shares = (df["ShOutFY"] - df["TrShFY"]).clip(lower=1)
feat["market_cap"] = (df["ADJ_CLOSE"] * float_shares).apply(
    lambda x: np.log1p(x) if pd.notna(x) and x > 0 else np.nan)

# 3-6. 正確版サプライズ（前回発表時の会社予想ベース）
# prev_FSales/prev_Sales = 前四半期発表時の通期予想/累積実績
# implied_q = (prev_FSales - prev_Sales) / 前回時点の残りQ数
_PREV_REMAINING = {"2Q": 3, "3Q": 2, "FY": 1}
prev_remaining = df["CurPerType"].map(_PREV_REMAINING)

prev_implied_q_sales = (df["prev_FSales"] - df["prev_Sales"]) / prev_remaining.replace(0, np.nan)
prev_implied_q_op    = (df["prev_FOP"]    - df["prev_OP"])    / prev_remaining.replace(0, np.nan)
prev_implied_q_np    = (df["prev_FNP"]    - df["prev_NP"])    / prev_remaining.replace(0, np.nan)

# Q2/Q3/FY: 正確版、Q1: 従来近似版にフォールバック
has_prev = df["prev_FSales"].notna()
feat["surprise_sales_accurate"] = np.where(
    has_prev, safe_ratio(df["Q_Sales"], prev_implied_q_sales),
    feat["surprise_sales_vs_forecast"])
feat["surprise_op_accurate"] = np.where(
    has_prev, safe_ratio(df["Q_OP"], prev_implied_q_op),
    feat["surprise_op_vs_forecast"])
feat["surprise_np_accurate"] = np.where(
    has_prev, safe_ratio(df["Q_NP"], prev_implied_q_np),
    feat["surprise_np_vs_forecast"])

# 3-7. 前年同期比（YoY）
feat["yoy_sales"] = safe_ratio(df["Q_Sales"], df["PrevY_Q_Sales"])
feat["yoy_op"]    = safe_ratio(df["Q_OP"],    df["PrevY_Q_OP"])
feat["yoy_np"]    = safe_ratio(df["Q_NP"],    df["PrevY_Q_NP"])

# 3-8. 同時アナウンス特徴量
feat["has_buyback"] = df["has_buyback"].astype(int)
feat["buyback_pct"] = df["buyback_pct"]
feat["has_stock_split"] = df["has_stock_split"].astype(int)
feat["split_ratio"] = df["split_ratio"]
feat["has_yutai"] = df["has_yutai"].astype(int)
feat["yutai_score"] = df["yutai_score"]  # +1=拡充/新設, -1=改悪/廃止, 0=不明

# 3-9. 事前修正フラグ（織り込み度合いの代理変数）
feat["has_prior_revision"]     = df["has_prior_revision"].astype(int)
feat["has_prior_div_revision"] = df["has_prior_div_revision"].astype(int)

feat["quarter"]     = df["CurPerType"]
feat["industry_33"] = df["INDUSTRY_33_CATEGORY"]
feat["size_cat"]    = df["SIZE_CATEGORY"]

feat["price_date"]   = df["price_date"]
feat["is_intraday"]  = df["is_intraday"]
feat["label_close_return"] = df["LABEL_CLOSE_RETURN"]
feat["label_open_return"]  = df["LABEL_OPEN_RETURN"]

print(f"特徴量DataFrame: {feat.shape}")
print("\nサプライズ欠損率（従来近似 / 正確版）:")
print(pd.DataFrame({
    "従来": feat[["surprise_sales_vs_forecast","surprise_op_vs_forecast","surprise_np_vs_forecast"]].isnull().mean().values,
    "正確版": feat[["surprise_sales_accurate","surprise_op_accurate","surprise_np_accurate"]].isnull().mean().values,
}, index=["sales","op","np"]))
print(f"\nYoY欠損率: {feat[['yoy_sales','yoy_op','yoy_np']].isnull().mean().to_dict()}")
print(f"自社株買いあり: {feat['has_buyback'].sum()} 件")
print(f"株式分割あり: {feat['has_stock_split'].sum()} 件")
print(f"優待変更あり: {feat['has_yutai'].sum()} 件 (拡充/改悪)")
print(f"事前業績修正あり: {feat['has_prior_revision'].sum()} 件")
print(f"事前配当修正あり: {feat['has_prior_div_revision'].sum()} 件")
print(f"ガイダンス修正あり: {feat["has_guidance_revision"].sum()} 件")
print(f"進捗率(営業利益)中央値: {feat["progress_op"].median():.1%}")
feat.head(5)

## 3-b. 特徴量 GCS 保存 / 読み込み

In [ ]:
import subprocess

GCS_BUCKET    = 'gs://stock_data_1930932'
GCS_FEATURES  = f'{GCS_BUCKET}/earnings_model/earnings_reaction_features'
GCS_MODELS    = f'{GCS_BUCKET}/earnings_model/models'
GCS_REPORTS   = f'{GCS_BUCKET}/earnings_model/reports'


def save_features_to_gcs(df: pd.DataFrame, date_from: str, date_to: str) -> str:
    """特徴量DataFrameをCSVでGCSにアップロード."""
    filename  = f'{date_from}_{date_to}.csv'
    tmp_path  = f'/content/{filename}'
    gcs_path  = f'{GCS_FEATURES}/{filename}'

    df.to_csv(tmp_path, index=False)
    result = subprocess.run(['gcloud', 'storage', 'cp', tmp_path, gcs_path],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[ERROR] GCSアップロード失敗:\n{result.stderr}')
        return ''
    os.remove(tmp_path)
    print(f'GCS保存完了: {gcs_path}')
    return gcs_path


def load_features_from_gcs(date_from: str, date_to: str) -> pd.DataFrame:
    """GCSから特徴量CSVをダウンロードして DataFrame で返す."""
    filename = f'{date_from}_{date_to}.csv'
    tmp_path = f'/content/{filename}'
    gcs_path = f'{GCS_FEATURES}/{filename}'

    result = subprocess.run(['gcloud', 'storage', 'cp', gcs_path, tmp_path],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[ERROR] GCSダウンロード失敗:\n{result.stderr}')
        return pd.DataFrame()
    df = pd.read_csv(tmp_path)
    df['DiscDate'] = pd.to_datetime(df['DiscDate']).dt.date
    os.remove(tmp_path)
    print(f'GCS読み込み完了: {gcs_path}  ({len(df)} 件)')
    return df


def list_features_on_gcs() -> None:
    """GCS上の特徴量ファイル一覧を表示."""
    result = subprocess.run(['gcloud', 'storage', 'ls', f'{GCS_FEATURES}/'],
                            capture_output=True, text=True)
    print(result.stdout if result.stdout else '(ファイルなし)')


# ── 保存実行（コメント解除で実行）──
save_features_to_gcs(feat, DATE_FROM, DATE_TO)

# ── 一覧確認 ──
list_features_on_gcs()

## 4. EDA

In [ ]:
# ── 4-1. ラベル分布 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

feat['label_close_return'].dropna().hist(bins=50, ax=axes[0])
axes[0].set_title('翌日終値騰落率 分布')
axes[0].set_xlabel('Return')
axes[0].axvline(0, color='red', linestyle='--')

feat['label_open_return'].dropna().hist(bins=50, ax=axes[1])
axes[1].set_title('翌日始値騰落率 分布')
axes[1].set_xlabel('Return')
axes[1].axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

print(feat['label_close_return'].describe())

In [ ]:
# ── 4-2. 四半期別リターン分布 ────────────────────────────────────
feat.boxplot(column='label_close_return', by='quarter', figsize=(10, 5))
plt.suptitle('')
plt.title('四半期別 翌日終値騰落率')
plt.xlabel('四半期種別')
plt.ylabel('Return')
plt.axhline(0, color='red', linestyle='--')
plt.show()

In [ ]:
# ── 4-3. 因子 × ラベル 散布図 ──────────────────────────────────
factor_cols = [
    'surprise_np_vs_forecast',
    'surprise_np_accurate',
    'surprise_odp_vs_consensus',
    'yoy_np',
    'guidance_fy_np',
    'dividend_surprise',
    'market_cap',
    'buyback_pct',
]

fig, axes = plt.subplots(1, len(factor_cols), figsize=(18, 4))
for ax, col in zip(axes, factor_cols):
    sub = feat[[col, 'label_close_return']].dropna()
    ax.scatter(sub[col], sub['label_close_return'], alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel(col)
    ax.set_ylabel('label_close_return')
    if len(sub) > 1:
        corr = sub.corr().iloc[0, 1]
        ax.set_title(f'r={corr:.3f} (n={len(sub)})')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4-4. 単変量IC（情報係数）一覧 ────────────────────────────────
from scipy import stats

ic_results = []
for col in factor_cols:
    sub = feat[[col, 'label_close_return']].dropna()
    if len(sub) < 10:
        continue
    corr, pval = stats.pearsonr(sub[col], sub['label_close_return'])
    spearman_r, spearman_p = stats.spearmanr(sub[col], sub['label_close_return'])
    ic_results.append({
        'factor': col, 'n': len(sub),
        'pearson_r': corr, 'pearson_p': pval,
        'spearman_r': spearman_r, 'spearman_p': spearman_p,
    })

if ic_results:
    df_ic = pd.DataFrame(ic_results).sort_values('spearman_r', ascending=False)
else:
    df_ic = pd.DataFrame(columns=['factor','n','pearson_r','pearson_p','spearman_r','spearman_p'])
    print('IC計算対象なし（各因子の有効サンプル n<10）')
df_ic

In [ ]:
# ── 4-5. セクター別リターン中央値 ────────────────────────────────
sector_ret = (
    feat.groupby('industry_33')['label_close_return']
    .agg(['median', 'mean', 'count'])
    .sort_values('median', ascending=False)
)
sector_ret.plot(y='median', kind='bar', figsize=(14, 4),
                title='セクター別 翌日終値騰落率（中央値）')
plt.axhline(0, color='red', linestyle='--')
plt.tight_layout()
plt.show()
sector_ret

## 6. クラスタ別精度分析（残差相関モデル）

ファクターモデル残差相関のクラスタラベルを結合し、predictスコアリング6因子の精度をクラスタ別に評価する。

In [ ]:
# ── 6-1. クラスタラベル結合 ────────────────────────────────────
if RUNTIME == 'colab':
    _cluster_dir = Path('/content/drive/MyDrive/analysis/factor_model')
else:
    _cluster_dir = Path(r'G:\マイドライブ\analysis\factor_model')

_cluster_frames = []
for y in [2024, 2025]:
    p = _cluster_dir / f'clusters_{y}.csv'
    if p.exists():
        _df = pd.read_csv(p, dtype={'TICKER': str})
        _df['year'] = y
        _cluster_frames.append(_df)
        print(f'clusters_{y}.csv: {len(_df)} 銘柄')

if not _cluster_frames:
    raise FileNotFoundError(f'クラスタCSVが見つかりません: {_cluster_dir}')

df_clusters = pd.concat(_cluster_frames, ignore_index=True)
print(f'クラスタデータ合計: {len(df_clusters)} 行')
display(df_clusters.head())

# ── feat に年を付与してクラスタ結合 ──
feat['_year'] = pd.to_datetime(feat['DiscDate']).dt.year
feat['_cluster_year'] = feat['_year'].clip(upper=2025)

feat['_ticker4'] = feat['Code'].astype(str).str[:4]
df_clusters['_ticker4'] = df_clusters['TICKER'].astype(str).str[:4]

feat_c = feat.merge(
    df_clusters[['_ticker4', 'CLUSTER', 'year']].rename(columns={'year': '_cluster_year'}),
    on=['_ticker4', '_cluster_year'],
    how='left'
)

n_matched = feat_c['CLUSTER'].notna().sum()
n_total = len(feat_c)
print(f'\nクラスタ結合: {n_matched}/{n_total} ({n_matched/n_total*100:.1f}%) の決算にクラスタラベル付与')
print(f'（未マッチ = 時価総額500億未満 or ETF/REIT）')


In [ ]:
# ── 6-2. スコアリング再現 + クラスタ別精度 ──────────────────────────
from scipy import stats

def _compute_score_eda(row: pd.Series) -> int:
    """predict 6因子スコアリングを再現."""
    score = 0
    q_map = {'1Q': 0.25, '2Q': 0.50, '3Q': 0.75}
    expected = q_map.get(row.get('quarter', ''), None)
    # F1: 進捗率
    if expected and pd.notna(row.get('surprise_op_vs_forecast')):
        if row['surprise_op_vs_forecast'] > 0.2: score += 1
        elif row['surprise_op_vs_forecast'] < -0.2: score -= 1
    # F2: ガイダンス修正
    if row.get('has_prior_revision') and pd.notna(row.get('surprise_op_accurate')):
        if row['surprise_op_accurate'] > 0.1: score += 1
        elif row['surprise_op_accurate'] < -0.1: score -= 1
    # F3: YoY OP
    if pd.notna(row.get('yoy_op')):
        if row['yoy_op'] > 0.3: score += 1
        elif row['yoy_op'] < -0.3: score -= 1
    # F4: コンセンサス乖離
    if pd.notna(row.get('surprise_odp_vs_consensus')):
        if row['surprise_odp_vs_consensus'] > 0.1: score += 1
        elif row['surprise_odp_vs_consensus'] < -0.1: score -= 1
    # F5: 翌期見通し（FYのみ）
    if row.get('quarter') == 'FY' and pd.notna(row.get('guidance_fy_op')):
        if row['guidance_fy_op'] > 0.1: score += 2
        elif row['guidance_fy_op'] < -0.1: score -= 2
    # F6: 出尽くしリスク（3Q）
    if row.get('quarter') == '3Q':
        pv = row.get('surprise_op_vs_forecast', None)
        if pd.notna(pv) and pv > 0.15 and not row.get('has_prior_revision', False):
            score -= 2
    return score

# クラスタ付きデータのみ（大型株）
fc = feat_c[feat_c['CLUSTER'].notna()].copy()
fc['score'] = fc.apply(_compute_score_eda, axis=1)

# 方向判定
fc['pred_dir'] = fc['score'].apply(lambda s: 1 if s > 0 else (-1 if s < 0 else 0))
fc['actual_dir'] = fc['label_close_return'].apply(
    lambda r: (1 if r > 0.005 else (-1 if r < -0.005 else 0)) if pd.notna(r) else None
)
fc['dir_match'] = fc['pred_dir'] == fc['actual_dir']

valid = fc[fc['actual_dir'].notna()]
overall_acc = valid['dir_match'].mean()
overall_corr = valid[['score', 'label_close_return']].corr().iloc[0, 1]
print(f'=== 全体（クラスタ付き大型株のみ）===')
print(f'件数: {len(valid)}, 方向一致率: {overall_acc:.3f}, スコアxリターン相関: {overall_corr:.3f}')
print()

# ── クラスタ別精度 ──
cluster_stats = []
for cid, grp in valid.groupby('CLUSTER'):
    n = len(grp)
    acc = grp['dir_match'].mean()
    corr = grp[['score', 'label_close_return']].corr().iloc[0, 1] if n >= 5 else None
    mean_ret = grp['label_close_return'].mean()
    top_ind = grp['industry_33'].value_counts().index[0] if 'industry_33' in grp.columns else ''
    cluster_stats.append({
        'cluster': int(cid), 'n': n,
        'dir_accuracy': round(acc, 3),
        'score_ret_corr': round(corr, 3) if corr is not None else None,
        'mean_return': round(mean_ret, 4),
        'top_industry': top_ind,
    })

df_cstats = pd.DataFrame(cluster_stats).sort_values('score_ret_corr', ascending=False, na_position='last')
print('=== クラスタ別精度（スコアxリターン相関 降順）===')
display(df_cstats)

# ── 可視化 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_plot = df_cstats[df_cstats['score_ret_corr'].notna()].sort_values('cluster')

axes[0].bar(df_plot['cluster'].astype(str), df_plot['dir_accuracy'], color='steelblue')
axes[0].axhline(y=overall_acc, color='red', linestyle='--', label=f'全体 {overall_acc:.3f}')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Direction Accuracy')
axes[0].set_title('クラスタ別 方向一致率')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(df_plot['cluster'].astype(str), df_plot['score_ret_corr'], color='darkorange')
axes[1].axhline(y=overall_corr, color='red', linestyle='--', label=f'全体 {overall_corr:.3f}')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Score-Return Correlation')
axes[1].set_title('クラスタ別 スコアxリターン相関')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\n=== 相関が高いクラスタ Top5 ===')
display(df_cstats.head(5))
print('\n=== 相関が低い/負のクラスタ Top5 ===')
display(df_cstats.tail(5))


## 5. TODO / 次のステップ

### 特徴量精度向上
- [x] ~~前回発表時の会社予想を取得~~ → `surprise_*_accurate`（Q1はフォールバック）
- [ ] **コンセンサスQ単独変換の設計**
  - STOCK.CONSENSUS の QUARTER カバレッジを銘柄別に集計してパターンを確認
  - 全Q揃っていれば差分計算可能。欠損Qが多ければ FY値÷4 等の近似も検討
- [x] ~~前年同期比（YoY）特徴量~~ → `yoy_sales`, `yoy_op`, `yoy_np`
- [ ] **事前修正フラグ**（DiscDate より前に同 Code の修正済みレコードが存在するか）
- [x] ~~自社株買いフラグ~~ → `has_buyback` / `buyback_pct`

### モデル検証
- [ ] データ期間を拡大して IC の安定性を確認
- [ ] 多変量モデル（ロジスティック回帰 / XGBoost）で予測力を検証
- [ ] 四半期×セクターの交互作用を確認